In [1]:
from spark_session import get_spark
spark = get_spark("notebook-explore")
spark.sql("SELECT COUNT(*) AS total_rows FROM local.booking.rental_property").show()

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.sedona#sedona-spark-shaded-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-19c9bf46-f5a0-4cca-8b50-e560b125b008;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.9.1 in central
	found org.apache.sedona#sedona-spark-shaded-3.5_2.12;1.6.1 in central
:: resolution report :: resolve 313ms :: artifacts dl 16ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.9.1 from central in [default]
	org.apache.sedona#sedona-spark-shaded-3.5_2.12;1.6.1 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	-----------------------------------

+----------+
|total_rows|
+----------+
|      2004|
+----------+



In [2]:
df = spark.sql("""
    SELECT external_id, property_name, city, country, star_rating,
           review_score, price, currency, is_published, latlon
    FROM local.booking.rental_property
    LIMIT 20
""")
df.toPandas()

,external_id,property_name,city,country,star_rating,review_score,price,currency,is_published,latlon
0,BC-1310393,Comfort Inn & Suites Caldwell,Caldwell,United States of America,5.0,8.80,247.78,USD,True,SRID=4326;POINT (-81.556727 39.80332)
1,BC-16745649,Villa Confidentielle Opale,Merlimont,France,NaN,None,None,USD,True,SRID=4326;POINT (1.607167 50.454845)
2,BC-15155564,Penthouse 60m from the Sea in Meia Praia PAT0402,Itapema,Brazil,4.0,10.00,297.40,USD,True,SRID=4326;POINT (-48.595041 -27.135939)
3,BC-9104459,Finesi Apartments 2,Ohrid,North Macedonia,3.0,9.40,80.49,USD,True,SRID=4326;POINT (20.795966 41.122966)
4,BC-16723909,Bamboo Khiri,Sathani Rot Fai,Thailand,3.0,None,100.04,USD,True,SRID=4326;POINT (99.798473 11.805727)
5,BC-16623725,Casa Vacanze Il Gallo,Trecchina,Italy,NaN,10.00,185.79,USD,True,SRID=4326;POINT (15.768254 40.024851)
6,BC-10841827,Ferienwohnung Hausboot an der Lagune Zwischend...,Büsum,Germany,3.0,7.80,182.10,USD,True,SRID=4326;POINT (8.846478 54.133336)
7,BC-2639100,Apartments ALBA Podstrana Familien-und Individ...,Podstrana,Croatia,4.0,9.60,178.79,USD,True,SRID=4326;POINT (16.54166 43.496117)
8,BC-13815719,"Résidence Althea - Centre d'Arcachon, emplacem...",Arcachon,France,NaN,None,None,USD,True,SRID=4326;POINT (-1.173959 44.661709)
9,BC-7613700,Residence Inn by Marriott Wenatchee,Wenatchee,United States of America,3.0,9.10,567.00,USD,True,SRID=4326;POINT (-120.318939 47.441508)


In [3]:
spark.sql("""
    SELECT external_id, property_name, ST_AsText(ST_GeomFromEWKT(latlon)) AS point
    FROM local.booking.rental_property
    WHERE latlon IS NOT NULL
    LIMIT 10
""").toPandas()

,external_id,property_name,point
0,BC-1310393,Comfort Inn & Suites Caldwell,POINT (-81.556727 39.80332)
1,BC-16745649,Villa Confidentielle Opale,POINT (1.607167 50.454845)
2,BC-15155564,Penthouse 60m from the Sea in Meia Praia PAT0402,POINT (-48.595041 -27.135939)
3,BC-9104459,Finesi Apartments 2,POINT (20.795966 41.122966)
4,BC-16723909,Bamboo Khiri,POINT (99.798473 11.805727)
5,BC-16623725,Casa Vacanze Il Gallo,POINT (15.768254 40.024851)
6,BC-10841827,Ferienwohnung Hausboot an der Lagune Zwischend...,POINT (8.846478 54.133336)
7,BC-2639100,Apartments ALBA Podstrana Familien-und Individ...,POINT (16.54166 43.496117)
8,BC-13815719,"Résidence Althea - Centre d'Arcachon, emplacem...",POINT (-1.173959 44.661709)
9,BC-7613700,Residence Inn by Marriott Wenatchee,POINT (-120.318939 47.441508)


In [4]:
spark.sql("""
    SELECT country, COUNT(*) AS n, ROUND(AVG(review_score), 2) AS avg_score
    FROM local.booking.rental_property
    GROUP BY country
    ORDER BY n DESC
""").toPandas()

,country,n,avg_score
0,Germany,177,8.33
1,United States of America,177,8.49
2,France,176,8.75
3,Brazil,130,8.90
4,United Kingdom,119,8.87
...,...,...,...
110,Luxembourg,1,None
111,"Moldova, Republic of",1,None
112,Solomon Islands,1,None
113,Bahrain,1,1.00


In [7]:
from snapshot_diff import diff_snapshots

snaps = spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM local.booking.rental_property.snapshots
    ORDER BY committed_at
""").collect()

for s in snaps:
    print(s["snapshot_id"], s["committed_at"], s["operation"])

465267461514840534 2026-07-16 09:12:41.729000 append
6667871004606134109 2026-07-17 09:33:36.429000 overwrite


In [8]:
old_id = snaps[-2]["snapshot_id"]
new_id = snaps[-1]["snapshot_id"]

changed = diff_snapshots(spark, old_id, new_id)

26/07/17 09:35:02 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


1 row(s) changed between snapshot 465267461514840534 and 6667871004606134109
{'feed_provider_id': '1976395', 'property_name': '4 star holiday home in Ulfborg', 'changed_fields': ['occupancy', 'max_occupancy']}
